In [16]:
#import libraries

import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings

In [2]:
#load and merge data

dec_df = pd.read_csv('/Users/tobyzhang/Desktop/idxexchange/data/CRMLSSold202412.csv')
jan_df = pd.read_csv('/Users/tobyzhang/Desktop/idxexchange/data/CRMLSSold202501_filled.csv')
feb_df = pd.read_csv('/Users/tobyzhang/Desktop/idxexchange/data/CRMLSSold202502.csv')
mar_df = pd.read_csv('/Users/tobyzhang/Desktop/idxexchange/data/CRMLSSold202503.csv')
apr_df = pd.read_csv('/Users/tobyzhang/Desktop/idxexchange/data/CRMLSSold202504.csv')
may_df = pd.read_csv('/Users/tobyzhang/Desktop/idxexchange/data/CRMLSSold202505.csv')
df_combined = pd.concat([dec_df, jan_df, feb_df, mar_df, apr_df, may_df], ignore_index=True)

In [3]:
#filter for residential, single family, and CA

df_filtered = df_combined[
    (df_combined['PropertyType'] == 'Residential') &
    (df_combined['PropertySubType'] == 'SingleFamilyResidence') &
    (df_combined['StateOrProvince'] == 'CA')
]



In [5]:
#define features and target

features = [
    "LivingArea", "LotSizeArea", "GarageSpaces", "BathroomsTotalInteger",
    "BedroomsTotal", "Stories", "YearBuilt", "Latitude", "Longitude",
    "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "ParkingTotal",
    "FireplaceYN", "CountyOrParish", "City"
]
target = "ClosePrice"
df_model = df_filtered[features + [target]].dropna(subset=[target])

In [6]:
#  Split data into training and testing sets
X = df_model[features]
y = df_model[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# 3. Preprocessing
categorical_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "CountyOrParish", "City"]
numeric_cols = [col for col in features if col not in categorical_cols]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [9]:
# 4. Log-transform target
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [10]:
# 5. Define and train each model
results = {}

In [11]:
# Ridge Regression
ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1.0, random_state=42))
])
ridge_pipe.fit(X_train, y_train_log)
ridge_pred_log = ridge_pipe.predict(X_test)
ridge_pred = np.expm1(ridge_pred_log)
results["Ridge"] = {
    "RMSE": np.sqrt(mean_squared_error(y_test, ridge_pred)),
    "R2": r2_score(y_test, ridge_pred)
}

In [12]:
# Lasso Regression
lasso_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Lasso(alpha=0.1, random_state=42, max_iter=10000))
])
lasso_pipe.fit(X_train, y_train_log)
lasso_pred_log = lasso_pipe.predict(X_test)
lasso_pred = np.expm1(lasso_pred_log)
results["Lasso"] = {
    "RMSE": np.sqrt(mean_squared_error(y_test, lasso_pred)),
    "R2": r2_score(y_test, lasso_pred)
}


In [13]:
# Decision Tree
tree_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", DecisionTreeRegressor(max_depth=10, random_state=42))
])
tree_pipe.fit(X_train, y_train_log)
tree_pred_log = tree_pipe.predict(X_test)
tree_pred = np.expm1(tree_pred_log)
results["Decision Tree"] = {
    "RMSE": np.sqrt(mean_squared_error(y_test, tree_pred)),
    "R2": r2_score(y_test, tree_pred)
}

In [14]:
# XGBoost
xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(objective="reg:squarederror", random_state=42))
])
xgb_pipe.fit(X_train, y_train_log)
xgb_pred_log = xgb_pipe.predict(X_test)
xgb_pred = np.expm1(xgb_pred_log)
results["XGBoost"] = {
    "RMSE": np.sqrt(mean_squared_error(y_test, xgb_pred)),
    "R2": r2_score(y_test, xgb_pred)
}

In [15]:
# 6. Display results
results_df = pd.DataFrame(results).T
print(results_df)


                       RMSE            R2
Ridge          2.455027e+08 -1.853375e+03
Lasso          1.160139e+10 -4.140998e+06
Decision Tree  5.529960e+06  5.913168e-02
XGBoost        5.533404e+06  5.795946e-02
